In [5]:
import numpy as np
import pandas as pd
import re


reviews = pd.read_csv("../data/restaurant_reviews.csv")
metadata = pd.read_csv("../data/restaurant_metadata.csv")
reviews_clean = reviews.copy()
metadata_clean = metadata.copy()

print("Original shape:", reviews.shape)
print("Working copy shape:", reviews_clean.shape)


Original shape: (10000, 7)
Working copy shape: (10000, 7)


In [6]:
print(reviews_clean.isnull().sum())

Restaurant     0
Reviewer      38
Review        45
Rating        38
Metadata      38
Time          38
Pictures       0
dtype: int64


In [10]:
print(reviews_clean[reviews_clean["Review"].isnull()].head())


           Restaurant                Reviewer Review Rating  \
2360             Amul        Lakshmi Narayana    NaN      5   
5799     Being Hungry                   Surya    NaN      5   
6449  Hyderabad Chefs          Madhurimanne97    NaN      5   
6489  Hyderabad Chefs                  Harsha    NaN      5   
7954     Olive Garden  ARUGULLA PRAVEEN KUMAR    NaN      3   

                     Metadata             Time  Pictures  
2360                0 Reviews  7/29/2018 18:00         0  
5799  4 Reviews , 4 Followers  7/19/2018 23:55         0  
6449                 1 Review  7/23/2018 16:29         0  
6489                 1 Review   7/8/2018 21:19         0  
7954    1 Review , 1 Follower   8/9/2018 23:25         0  


In [9]:
print(reviews_clean[reviews_clean["Rating"].isnull()].head())

               Restaurant Reviewer Review Rating Metadata Time  Pictures
8777  American Wild Wings      NaN    NaN    NaN      NaN  NaN         0
8778  American Wild Wings      NaN    NaN    NaN      NaN  NaN         0
8779  American Wild Wings      NaN    NaN    NaN      NaN  NaN         0
8780  American Wild Wings      NaN    NaN    NaN      NaN  NaN         0
8781  American Wild Wings      NaN    NaN    NaN      NaN  NaN         0


In [13]:
reviews_clean[
    reviews_clean["Review"].notnull() &
    reviews_clean["Rating"].notnull()
].shape

(9955, 7)

In [14]:
reviews_clean[
    reviews_clean["Review"].isnull() &
    reviews_clean["Rating"].isnull()
].shape

(38, 7)

In [15]:
reviews_clean[
    reviews_clean["Review"].isnull() &
    reviews_clean["Rating"].notnull()
].shape

(7, 7)

In [16]:
reviews_clean[
    reviews_clean["Rating"].isnull() &
    reviews_clean["Review"].notnull()
].shape

(0, 7)

In [19]:
# Cleaning operation
reviews_clean = reviews_clean.dropna(
    subset = ["Review", "Rating"],
    how = "all"
)
# how = "all" means remove a row only when all specified colums are missing
# how = "any" means remove a row if even one specified column is missing

# verifying the results:
print(reviews_clean.shape)
print(reviews_clean.isnull().sum())


(9962, 7)
Restaurant    0
Reviewer      0
Review        7
Rating        0
Metadata      0
Time          0
Pictures      0
dtype: int64


In [20]:
# Duplicate investigation

reviews_clean.duplicated().sum()

0

In [22]:
reviews.duplicated().sum()

36

In [23]:
reviews_clean.duplicated(
    subset = ["Restaurant","Reviewer","Review"]
).sum()

6

In [24]:
reviews[reviews.duplicated(keep = False)].head(20)

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
8777,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8778,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8779,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8780,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8781,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8782,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8783,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8784,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8785,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0
8786,American Wild Wings,NaN,NaN,NaN,NaN,NaN,0


In [25]:
reviews_clean[
    reviews_clean.duplicated(
        subset=["Restaurant", "Reviewer", "Review"],
        keep=False
    )
].sort_values(
    by=["Restaurant", "Reviewer"]
)

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
2349,Amul,Rahul Nayak,taste was good,5,"20 Reviews , 6 Followers",8/3/2018 17:40,0
2352,Amul,Rahul Nayak,taste was good,5,"20 Reviews , 6 Followers",7/31/2018 19:06,0
5750,Being Hungry,Prite,good,5,"5 Reviews , 1 Follower",8/7/2018 23:50,0
5766,Being Hungry,Prite,good,5,"5 Reviews , 1 Follower",7/28/2018 0:51,0
5787,Being Hungry,Rajaboinaprashanth333,very good,5,"7 Reviews , 1 Follower",7/21/2018 23:43,0
5798,Being Hungry,Rajaboinaprashanth333,very good,5,"7 Reviews , 1 Follower",7/20/2018 0:14,0
5762,Being Hungry,Thiru Yadav,very good,5,4 Reviews,7/29/2018 6:59,0
5772,Being Hungry,Thiru Yadav,very good,5,4 Reviews,7/27/2018 1:59,0
3529,Green Bawarchi Restaurant,Nalinikanta Tripathy,very good,5,"5 Reviews , 1 Follower",10/24/2018 20:26,0
3537,Green Bawarchi Restaurant,Nalinikanta Tripathy,very good,5,"5 Reviews , 1 Follower",9/30/2018 13:05,0


In [26]:
reviews_clean["Rating"].value_counts(dropna=False)

Rating
5       3832
4       2373
1       1735
3       1193
2        684
4.5       69
3.5       47
2.5       19
1.5        9
Like       1
Name: count, dtype: int64

In [27]:
numeric_ratings = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

invalid_ratings = reviews_clean.loc[
    numeric_ratings.isna(),
    "Rating"
]

invalid_ratings.value_counts(dropna=False)

Rating
Like    1
Name: count, dtype: int64

In [29]:
reviews_clean["Rating_numeric"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

print(reviews_clean["Rating_numeric"].dtype)

print(reviews_clean["Rating_numeric"].isnull().sum())

print(reviews_clean[
    reviews_clean["Rating_numeric"].isnull()
][["Review", "Rating"]])

float64
1
                                                 Review Rating
7601  One of the best pizzas to try. It served with ...   Like


In [30]:
def clean_review(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove emojis and special characters
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


sample_reviews = [
    "AMAZING FOOD!!! 😍😍",
    "Delivery took 45 mins!!!",
    "Visit https://example.com for offers",
    "@ramana Food was GREAT!!!",
    "Food     quality was good..."
]

for review in sample_reviews:
    print("Original:", review)
    print("Cleaned :", clean_review(review))
    print("-" * 40)

Original: AMAZING FOOD!!! 😍😍
Cleaned : amazing food
----------------------------------------
Original: Delivery took 45 mins!!!
Cleaned : delivery took 45 mins
----------------------------------------
Original: Visit https://example.com for offers
Cleaned : visit for offers
----------------------------------------
Original: @ramana Food was GREAT!!!
Cleaned : food was great
----------------------------------------
Original: Food     quality was good...
Cleaned : food quality was good
----------------------------------------


In [31]:
reviews_clean["Review_clean"] = (
    reviews_clean["Review"].apply(clean_review)
)

In [32]:
reviews_clean[
    ["Review", "Review_clean"]
].sample(15, random_state=42)

,Review,Review_clean
6448,good,good
6831,My exp with the place was a bit different from...,my exp with the place was a bit different from...
8451,"awesome place to be during evenings, must try ...",awesome place to be during evenings must try k...
39,I came here with my parents. We ordered for Th...,i came here with my parents we ordered for tha...
7458,Very very disappointed.\nFood taste was not up...,very very disappointed food taste was not up t...
3776,good,good
2069,The food was prepared quickly and was also del...,the food was prepared quickly and was also del...
4347,"Had one of those afternoon sweet cravings, so ...",had one of those afternoon sweet cravings so t...
3936,Good ambience and service. If you are looking ...,good ambience and service if you are looking f...
568,Situated at end of the city. Very Spacious and...,situated at end of the city very spacious and ...


In [33]:
print("Total reviews:", len(reviews_clean))

print("Missing cleaned reviews:",
      reviews_clean["Review_clean"].isnull().sum())

print("Empty cleaned reviews:",
      (reviews_clean["Review_clean"] == "").sum())

Total reviews: 9962
Missing cleaned reviews: 0
Empty cleaned reviews: 17


In [34]:
empty_reviews = reviews_clean[
    reviews_clean["Review_clean"] == ""
]

print("Empty cleaned records:", len(empty_reviews))

print(
    empty_reviews[
        ["Restaurant", "Review", "Rating"]
    ].to_string()
)

Empty cleaned records: 17
                                 Restaurant Review Rating
1583                                    KFC      😄      5
2360                                   Amul    NaN      5
3784                    Kritunga Restaurant      .      5
4573                            Owm Nom Nom      💝      5
4582                            Owm Nom Nom      😊      5
5280                      Biryanis And More      .      5
5799                           Being Hungry    NaN      5
6274  Shree Santosh Dhaba Family Restaurant      .      5
6281  Shree Santosh Dhaba Family Restaurant      .      5
6449                        Hyderabad Chefs    NaN      5
6489                        Hyderabad Chefs    NaN      5
6659                           Aromas@11SIX     ☺😊      5
7954                           Olive Garden    NaN      3
8228                     Al Saba Restaurant    NaN      5
8830                         Domino's Pizza      .      4
8844                         Domino's Pizza   

In [35]:
print(
    "Originally missing reviews:",
    empty_reviews["Review"].isnull().sum()
)

print(
    "Originally non-empty reviews:",
    empty_reviews["Review"].notnull().sum()
)

Originally missing reviews: 7
Originally non-empty reviews: 10


In [36]:
reviews_clean = reviews_clean[
    reviews_clean["Review_clean"].str.strip() != ""
].copy()

print("Shape after cleaning:", reviews_clean.shape)

print(
    "Empty reviews remaining:",
    (reviews_clean["Review_clean"] == "").sum()
)

Shape after cleaning: (9945, 9)
Empty reviews remaining: 0


In [40]:
import re

def extract_delivery_time(review):
    if not isinstance(review, str):
        return None, None

    text = review.lower()

    # Hours: 1 hour, 2 hrs, 1.5 hours
    hour_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|hr)\b", text)

    if hour_match:
        hours = float(hour_match.group(1))
        return hour_match.group(), hours * 60

    # Minutes: 45 mins, 30 minutes, 90 min
    minute_match = re.search(r"(\d+)\s*(?:mins?|minutes?)\b", text)

    if minute_match:
        minutes = int(minute_match.group(1))
        return minute_match.group(), minutes

    return None, None

sample_reviews = [
    "Delivery took 45 mins",
    "Delivered in 30 minutes",
    "Delivery took 1 hour",
    "Food was delicious",
    "It took 90 minutes to arrive",
    "Delivery was very late",
    "Delivery took 1.5 hours"
]

for review in sample_reviews:
    matched_text, minutes = extract_delivery_time(review)

    if minutes is not None:
        print(f"{review} -> Matched: {matched_text}, Minutes: {minutes}")
    else:
        print(f"{review} -> No delivery time found")

Delivery took 45 mins -> Matched: 45 mins, Minutes: 45
Delivered in 30 minutes -> Matched: 30 minutes, Minutes: 30
Delivery took 1 hour -> Matched: 1 hour, Minutes: 60.0
Food was delicious -> No delivery time found
It took 90 minutes to arrive -> Matched: 90 minutes, Minutes: 90
Delivery was very late -> No delivery time found
Delivery took 1.5 hours -> Matched: 1.5 hours, Minutes: 90.0


In [41]:
# Detect delivery complaints 
def detect_delivery_complaint(review):
    if not isinstance(review, str):
        return 0

    text = review.lower()

    delivery_complaint_patterns = [
        r"\blate\b",
        r"\bdelay(?:ed|s)?\b",
        r"\bnot delivered\b",
        r"\bno delivery\b",
        r"\btook (?:too )?(?:long|much time)\b",
        r"\bnever arrived\b",
        r"\bcancelled\b"
    ]

    return int(any(re.search(pattern, text) for pattern in delivery_complaint_patterns))


for review in sample_reviews:
    print(review, "-> Delivery complaint:", detect_delivery_complaint(review))

Delivery took 45 mins -> Delivery complaint: 0
Delivered in 30 minutes -> Delivery complaint: 0
Delivery took 1 hour -> Delivery complaint: 0
Food was delicious -> Delivery complaint: 0
It took 90 minutes to arrive -> Delivery complaint: 0
Delivery was very late -> Delivery complaint: 1
Delivery took 1.5 hours -> Delivery complaint: 0


In [46]:
# Apply to the dataset
def extract_delivery_time(review):
    if not isinstance(review, str):
        return None, None

    text = review.lower()

    hour_match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|hr)\b",
        text
    )

    if hour_match:
        hours = float(hour_match.group(1))

        if 0 < hours <= 24:
            return hour_match.group(), hours * 60

    minute_match = re.search(
        r"(\d+)\s*(?:mins?|minutes?)\b",
        text
    )

    if minute_match:
        minutes = int(minute_match.group(1))

        if 0 < minutes <= 600:
            return minute_match.group(), minutes

    return None, None

reviews_clean[
    ["Delivery_Time_Match", "Delivery_Time_Minutes"]
] = reviews_clean["Review_clean"].apply(
    lambda review: pd.Series(extract_delivery_time(review))
)

In [47]:
reviews_clean["Delivery_Time_Minutes"].notna().sum()

298

In [48]:
reviews_clean["Delivery_Time_Minutes"].describe()

count     298.000000
mean       89.238255
std       248.990569
min         3.000000
25%        15.000000
50%        30.000000
75%        48.750000
max      1440.000000
Name: Delivery_Time_Minutes, dtype: float64

In [49]:
# Next step: inspect high values
reviews_clean.loc[
    reviews_clean["Delivery_Time_Minutes"] > 120,
    ["Review_clean", "Delivery_Time_Match", "Delivery_Time_Minutes", "Rating"]
].sort_values(
    "Delivery_Time_Minutes",
    ascending=False
).head(10)

,Review_clean,Delivery_Time_Match,Delivery_Time_Minutes,Rating
9621,follow my instagram hyderabadi daawat a new pl...,24 hours,1440.0,5
7228,i ordered a triple chocolate cake it is one of...,24hrs,1440.0,4
2810,i was promised to get a refund for an order th...,24 hours,1440.0,1
9219,visited the place at midnight the best thing i...,24hours,1440.0,4
9198,came here for some late night dinner and my ex...,24 hours,1440.0,1
4428,this ppl were very irresponsible there swiping...,24hrs,1440.0,1
7275,one of the premium bakers in gachibowli most o...,24hrs,1440.0,3.5
2552,we celebrated the a friends bday and it was go...,24 hours,1440.0,4
7202,it is one of the premium bakers in gachibowli ...,24 hours,1440.0,4
5632,i dont even want to beg 1 star today for dinne...,12 hours,720.0,1


In [62]:
def extract_delivery_time(review):
    if not isinstance(review, str):
        return None, None

    text = review.lower()

    delivery_context = re.search(
        r"(delivery|delivered|deliver|arrived|arrival|order|ordered)",
        text
    )

    if not delivery_context:
        return None, None

    # Handles: 1.5 hours, 1 5 hours, 2 hours
    hour_match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|hr)\b",
        text
    )

    if hour_match:
        hours = float(hour_match.group(1))

        if 0 < hours <= 6:
            return hour_match.group(), hours * 60

    # Handles: 1 5 hours -> 1.5 hours
    spaced_decimal_match = re.search(
        r"(\d)\s(\d)\s*(?:hours?|hrs?|hr)\b",
        text
    )

    if spaced_decimal_match:
        hours = float(
            spaced_decimal_match.group(1)
            + "."
            + spaced_decimal_match.group(2)
        )

        if 0 < hours <= 6:
            return spaced_decimal_match.group(), hours * 60

    # Handles: 45 mins, 30 minutes, 90 min
    minute_match = re.search(
        r"(\d+)\s*(?:mins?|minutes?)\b",
        text
    )

    if minute_match:
        minutes = int(minute_match.group(1))

        if 0 < minutes <= 180:
            return minute_match.group(), minutes

    return None, None

In [63]:
reviews_clean[
    ["Delivery_Time_Match", "Delivery_Time_Minutes"]
] = reviews_clean["Review_clean"].apply(
    lambda review: pd.Series(extract_delivery_time(review))
)

In [64]:
reviews_clean.loc[
    reviews_clean["Review_clean"].str.contains(
        r"1 5 hours|3 4hrs|2 5 hours|2 3 hours",
        case=False,
        na=False,
        regex=True
    ),
    ["Review_clean", "Delivery_Time_Match", "Delivery_Time_Minutes", "Rating"]
]

,Review_clean,Delivery_Time_Match,Delivery_Time_Minutes,Rating
576,very slow service it took us more than 2 5 hou...,None,NaN,2
3364,i am thinking where to start there were so man...,3 hours,180.0,1
3436,the only reason this place gets a two is becau...,None,NaN,2
4916,overrated restaurant with pathetic service and...,5 hours,300.0,1
6020,its almost like they do not want to get you yo...,5 hours,300.0,1
7414,food was delivered after 1 5 hours when the ap...,5 hours,300.0,1
9030,the food here was ok but the service was worst...,4hrs,240.0,2


In [65]:
reviews_clean["Delivery_Time_Minutes"].describe()

count    194.000000
mean      44.701031
std       49.610747
min        3.000000
25%       15.000000
50%       30.000000
75%       45.000000
max      300.000000
Name: Delivery_Time_Minutes, dtype: float64

In [66]:
reviews_clean.loc[
    reviews_clean["Delivery_Time_Minutes"] > 120,
    ["Review_clean", "Delivery_Time_Match", "Delivery_Time_Minutes", "Rating"]
].sort_values(
    "Delivery_Time_Minutes",
    ascending=False
).head(10)

,Review_clean,Delivery_Time_Match,Delivery_Time_Minutes,Rating
4916,overrated restaurant with pathetic service and...,5 hours,300.0,1
6020,its almost like they do not want to get you yo...,5 hours,300.0,1
7414,food was delivered after 1 5 hours when the ap...,5 hours,300.0,1
9030,the food here was ok but the service was worst...,4hrs,240.0,2
2887,poor customer support both me and zomato were ...,3 hours,180.0,1
3364,i am thinking where to start there were so man...,3 hours,180.0,1
9277,sometimes the most unexpected evenings can tur...,3 hours,180.0,5
